# Chapter 20 Guided To-Do Practice Labs

**Book Reference:** Chapter 20, Section 20.5 (Guided To-Do Practice)  
Run **Step 0** to activate `sharukh-tutor`, implement all 6 biostatistical and epidemiological functions, and pass each **function name (without parentheses!)** to `tutor.check(...)`. Use `tutor.hint("ch20_todo_XX", level=1..3)` whenever you want a progressive nudge.

In [ ]:
# @title Step 0: Click Play (▶) to Activate Interactive Hints & Auto-Checker
!pip install -q sharukh-tutor
import math
import numpy as np
import pandas as pd
from scipy import stats
import sharukh_tutor as tutor

tutor.activate(chapter="ch20", notebook="guided_todos")


---
## To-Do 20.1 (Proportion Estimation): `wilson_ci(successes, n, z=1.96)`

Implement `wilson_ci(successes, n, z=1.96)` to compute the sample proportion $\hat{p} = \text{successes} / n$ and its Wilson score confidence interval:
- Denominator: $d = 1 + z^2 / n$
- Center: $c = (\hat{p} + z^2 / (2n)) / d$
- Margin: $m = (z \sqrt{(\hat{p}(1 - \hat{p}) + z^2 / (4n)) / n}) / d$
- Return a 3-tuple of floats `(prop, low, high)` rounded to **4 decimal places** (clamping `low >= 0.0` and `high <= 1.0`).

In [ ]:
def wilson_ci(successes, n, z=1.96):
    """Return (prop, low, high) 95% Wilson CI rounded to 4 decimals."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    p = successes / n
    denom = 1.0 + (z ** 2) / n
    center = (p + (z ** 2) / (2.0 * n)) / denom
    margin = (
        z * math.sqrt((p * (1.0 - p) + (z ** 2) / (4.0 * n)) / n)
    ) / denom
    low = max(0.0, center - margin)
    high = min(1.0, center + margin)
    return (round(p, 4), round(low, 4), round(high, 4))
    ### END SOLUTION


In [ ]:
tutor.check("ch20_todo_01", wilson_ci)


---
## To-Do 20.2 (Parametric Comparison & Effect Size): `compare_two_groups_welch(group_a, group_b)`

Implement `compare_two_groups_welch(group_a, group_b)`:
- Run Welch's two-sample t-test via `stats.ttest_ind(a, b, equal_var=False)`.
- Compute pooled standard deviation from sample variances (`ddof=1`):
  $$s_p = \sqrt{\frac{(n_a - 1)s_a^2 + (n_b - 1)s_b^2}{n_a + n_b - 2}}$$
- Compute Cohen's $d = (\bar{x}_a - \bar{x}_b) / s_p$.
- Return `{"t_stat": round4, "p_value": round4, "cohens_d": round4, "significant_05": bool}`.

In [ ]:
def compare_two_groups_welch(group_a, group_b):
    """Run Welch's t-test and compute pooled Cohen's d."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    a = np.asarray(group_a, dtype=float)
    b = np.asarray(group_b, dtype=float)
    res = stats.ttest_ind(a, b, equal_var=False)
    n_a, n_b = len(a), len(b)
    var_a = float(np.var(a, ddof=1))
    var_b = float(np.var(b, ddof=1))
    pooled_sd = math.sqrt(
        ((n_a - 1) * var_a + (n_b - 1) * var_b) / (n_a + n_b - 2)
    )
    cohens_d = (float(np.mean(a)) - float(np.mean(b))) / pooled_sd
    return {
        "t_stat": round(float(res.statistic), 4),
        "p_value": round(float(res.pvalue), 4),
        "cohens_d": round(float(cohens_d), 4),
        "significant_05": bool(res.pvalue < 0.05),
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch20_todo_02", compare_two_groups_welch)


---
## To-Do 20.3 (Nonparametric Rank Test): `nonparametric_mannwhitney(group_a, group_b)`

Implement `nonparametric_mannwhitney(group_a, group_b)`:
- Run `stats.mannwhitneyu(group_a, group_b, alternative="two-sided")`.
- Return `{"u_stat": round2, "p_value": round4, "significant_05": bool}`.

In [ ]:
def nonparametric_mannwhitney(group_a, group_b):
    """Run a two-sided Mann-Whitney U test on two independent groups."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    res = stats.mannwhitneyu(group_a, group_b, alternative="two-sided")
    return {
        "u_stat": round(float(res.statistic), 2),
        "p_value": round(float(res.pvalue), 4),
        "significant_05": bool(res.pvalue < 0.05),
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch20_todo_03", nonparametric_mannwhitney)


---
## To-Do 20.4 (Contingency Table Audit): `chi2_contingency_audit(table_2x2)`

Implement `chi2_contingency_audit(table_2x2)`:
- Run Pearson's chi-square test via `stats.chi2_contingency(table_2x2, correction=False)`.
- Return `{"chi2": round4, "p_value": round4, "dof": int, "expected_min": round2}`.

In [ ]:
def chi2_contingency_audit(table_2x2):
    """Run Pearson chi-square (no Yates) and audit min expected count."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    chi2, p_val, dof, expected = stats.chi2_contingency(
        table_2x2, correction=False
    )
    return {
        "chi2": round(float(chi2), 4),
        "p_value": round(float(p_val), 4),
        "dof": int(dof),
        "expected_min": round(float(np.min(expected)), 2),
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch20_todo_04", chi2_contingency_audit)


---
## To-Do 20.5 (Epidemiological Effect Size): `odds_ratio_with_ci(a, b, c, d)`

Implement `odds_ratio_with_ci(a, b, c, d)` using Woolf's log-OR 95% confidence interval (`z = 1.96`):
- $\text{OR} = (a \cdot d) / (b \cdot c)$
- $\text{SE} = \sqrt{1/a + 1/b + 1/c + 1/d}$
- $\text{CI} = \exp(\ln(\text{OR}) \pm 1.96 \cdot \text{SE})$
- Return `{"or": round3, "ci_low": round3, "ci_high": round3}`.

In [ ]:
def odds_ratio_with_ci(a, b, c, d):
    """Compute crude Odds Ratio and Woolf 95% CI rounded to 3 decimals."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    or_val = (a * d) / (b * c)
    se = math.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    log_or = math.log(or_val)
    return {
        "or": round(or_val, 3),
        "ci_low": round(math.exp(log_or - 1.96 * se), 3),
        "ci_high": round(math.exp(log_or + 1.96 * se), 3),
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch20_todo_05", odds_ratio_with_ci)


---
## To-Do 20.6 (Multiple Testing Control): `benjamini_hochberg_adjust(p_values)`

Implement `benjamini_hochberg_adjust(p_values)`:
- Compute Benjamini-Hochberg FDR-adjusted $q$-values matching `stats.false_discovery_control(p_values, method="bh")`.
- Return a Python `list` of floats rounded to **4 decimal places** in the original input order (without mutating `p_values`).

In [ ]:
def benjamini_hochberg_adjust(p_values):
    """Return BH FDR-adjusted q-values as a 4-decimal Python list."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    q_vals = stats.false_discovery_control(p_values, method="bh")
    return [round(float(q), 4) for q in q_vals]
    ### END SOLUTION


In [ ]:
tutor.check("ch20_todo_06", benjamini_hochberg_adjust)


---
## Annotated Debugging Post-Mortem (Solution Notebook Only)

- **To-Do 20.1 (Wald vs. Wilson at Zero Events):** For `wilson_ci(0, 20)`, the naive Wald interval collapses to `[0.0, 0.0]`, falsely implying 100% certainty that prevalence is zero. The Wilson score interval yields `[0.0, 0.1611]`, accurately reflecting uncertainty in a small sample.
- **To-Do 20.2 (Student vs. Welch Default):** SciPy's `stats.ttest_ind` defaults to `equal_var=True` (Student's t-test). Always pass `equal_var=False` for Welch's t-test, which remains valid when group variances and sample sizes differ.
- **To-Do 20.4 (Yates' Continuity Correction Default):** `stats.chi2_contingency` applies Yates' continuity correction (`correction=True`) to 2x2 tables by default, which is overly conservative. Pass `correction=False` for standard Pearson chi-square, and inspect `expected_min < 5` to decide when to switch to `stats.fisher_exact`.

In [ ]:
tutor.summary()
